In [5]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from io import StringIO
import psycopg

In [ ]:
#!uv pip install psycopg[binary]

Using Python 3.14.3 environment at: c:\Users\Lenovo\Documents\.venv
Resolved 3 packages in 169ms
 Downloaded psycopg-binary
Prepared 1 package in 261ms
Installed 1 package in 24ms
 + psycopg-binary==3.3.3


Código de cada database de la API como constantes

In [6]:
EXTERNAL_NET_DEBT = "BPS/Q.N.AT+BE+HR+CY+EE+FI+FR+DE+GR+IE+IT+LV+LT+LU+MT+NL+PT+SK+SI+ES+I9.W1.S1.S1.LE.NE.FA._T.FNED._Z.EUR._T._X.N.ALL"
EXTERNAL_NET_DEBT_PERCENTAGE = "BPS/Q.N.AT+BE+HR+CY+EE+FI+FR+DE+GR+IE+IT+LV+LT+LU+MT+NL+PT+SK+SI+ES+I9.W1.S1.S1.LE.NE.FA._T.FNED._Z.EUR_R_B1GQ._T._X.N.ALL"
GROSS_EXTERNAL_DEBT = "BPS/Q.N.AT+BE+HR+CY+EE+FI+FR+DE+GR+IE+IT+LV+LT+LU+MT+NL+PT+SK+SI+ES+I9.W1.S1.S1.LE.L.FA._T.FGED._Z.EUR._T._X.N.ALL"
INFLATION = "ICP/M.DE+FR+ES+IT+NL+BE+AT+PT+FI+IE+GR+SK+SI+EE+LV+LT+LU+CY+MT+U2.N.000000+FOOD00+NRGY00+SERV00+GOODS00+CP0000+ALCBEV+TOBACC+CLTHES+HOUSNG+FURNSH+HEALTH+TRANS+COMMUN+RECREA+EDUCAT+RESTAU+MISCEL.4.ANR"
EXCHANGE_RATE = "EXR/D.BGN+CZK+DKK+HUF+PLN+RON+SEK+CHF+NOK+GBP+ISK+USD+JPY+AUD+CAD+NZD.EUR.SP00.A"

Definición de la función extractora

In [7]:
def api_extract_data(code_data, start_date=None):

    core_url = "https://data-api.ecb.europa.eu/service/data/"
    url = core_url + code_data

    if start_date is not None:
        url += f"?startPeriod={start_date}"

     # Headers to store the content negotiation setting for the format of the output file.
    headers = {
        "Accept": "application/vnd.sdmx.structurespecificdata+xml;version=2.1"
    }

    response = requests.get(url, headers=headers)

    return response

Aplicación de la función extractora a los datos de cambios de divisa

Creación de la tabla con los cambios de divisa para exportar directamente a PostgreSQL

In [8]:
data_exchange_rate_bce = api_extract_data(EXCHANGE_RATE, start_date = "2009-01-01")

In [22]:
def create_csv_exchange_rate(response):

    root = ET.fromstring(response.content)
    series = root.findall(".//{*}Series")

    buffer = StringIO()

    for serie in series:
        currency = serie.attrib.get("CURRENCY")
        values = serie.findall(".//{*}Obs")
        for value in values:
            time_period = value.attrib.get("TIME_PERIOD")
            data_value = value.attrib.get("OBS_VALUE")
            buffer.write(f"{time_period},{currency},{data_value}\n")

    buffer.seek(0)

    for i in range(10):
        print(buffer.readline())

    return buffer

buffer_exchange_rate_data es el csv listo para guardar en la base de datos

In [24]:
buffer_exchange_rate_data = create_csv_exchange_rate(data_exchange_rate_bce)

2009-01-01,AUD,NaN

2009-01-02,AUD,1.9884

2009-01-05,AUD,1.9189

2009-01-06,AUD,1.8727

2009-01-07,AUD,1.8831

2009-01-08,AUD,1.9357

2009-01-09,AUD,1.9371

2009-01-12,AUD,1.9478

2009-01-13,AUD,1.9874

2009-01-14,AUD,1.9769



In [ ]:
#buffer_exchange_rate_data.readline()

AttributeError: '_io.StringIO' object has no attribute 'describe'